In [ ]:
# The Sora API lets you generate videos from text prompts. 
# Video generation is asynchronous - you create a job, poll for completion, then download the result.

import sys
import time

# Create a video generation job using Sora
video = client.videos.create(
    model="sora-2",
    prompt="A timelapse of a flower blooming in a sunlit garden, cinematic quality",
)

print(f"Video generation started! ID: {video.id}")
print(f"Initial status: {video.status}")

# Poll for completion with a progress bar
bar_length = 30
while video.status in ("in_progress", "queued"):
    video = client.videos.retrieve(video.id)
    progress = getattr(video, "progress", 0)

    filled = int((progress / 100) * bar_length)
    bar = "=" * filled + "-" * (bar_length - filled)
    status_text = "Queued" if video.status == "queued" else "Processing"

    sys.stdout.write(f"\r{status_text}: [{bar}] {progress:.1f}%")
    sys.stdout.flush()
    time.sleep(5)

sys.stdout.write("\n")

if video.status == "failed":
    message = getattr(getattr(video, "error", None), "message", "Video generation failed")
    print(f"Error: {message}")
else:
    print("Video generation completed!")

    # Download the video
    content = client.videos.download_content(video.id, variant="video")
    content.write_to_file("generated_video.mp4")
    print("Saved to generated_video.mp4")